In [2]:
import subprocess
result = subprocess.run(['pip', 'install', 'transformers==5.8.0', 'accelerate', '-q', '--no-deps'], capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else "done")
print(result.stderr[-300:] if result.stderr else "")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 88.0 MB/s eta 0:00:00




In [3]:
!pip install -q "huggingface-hub>=1.5.0" tokenizers safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 10.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 76.4 MB/s eta 0:00:00:00:01


In [6]:
!pip install -q gradio gtts
!pip install -q openai-whisper
!apt-get install -q ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
typer-slim 0.24.0 requires typer>=0.24.0, but you have typer 0.23.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.6 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.


In [7]:
!pip install -q "typer==0.9.0" "click==8.1.7"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
huggingface-hub 1.14.0 requires typer>=0.20.0, but you have typer 0.9.0 which is incompatible.
typer-slim 0.24.0 requires typer>=0.24.0, but you have typer 0.9.0 which is incompatible.
google-adk 1.25.1 requires click<9.0.0,>=8.1.8, but you have click 8.1.7 which is incompatible.
gradio 5.50.0 requires typer<1.0,>=0.12, but you have typer 0.9.0 which is incompatible.


In [4]:
import kagglehub
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e4b-it")
print("Path:", path)

processor = AutoProcessor.from_pretrained(path)
model = AutoModelForImageTextToText.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("✅ Model loaded")

Path: /kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

✅ Model loaded


In [15]:
from IPython.display import display, HTML, Audio
import ipywidgets as widgets
import torch
from gtts import gTTS
from PIL import Image
import tempfile, io, whisper

# Load Whisper

whisper_model = whisper.load_model("base")


SYSTEM_PROMPT = """You are AidFirst, a calm medical first-response assistant helping untrained caregivers understand a patient's condition.

FIRST THING TO DO BEFORE ANYTHING ELSE:
- Detect what language the person is using
- If English → your ENTIRE response must be in English only
- If Hindi → your ENTIRE response must be in Hindi only
- If mixed → respond in mixed Hindi-English
- This rule overrides everything else. Never switch languages.

EXAMPLE:
Person says: "my uncle is vomiting" → You respond in ENGLISH
Person says: "dadi gir gayi" → You respond in HINDI
Person says: "uncle ko vomiting ho rahi hai" → You respond in HINDI

YOU ARE NOT A DOCTOR. You do not diagnose. You help people understand seriousness and next steps.

YOUR BEHAVIOUR CHANGES based on how much you know:

--- PHASE 1: First 1-3 messages (gathering information) ---
Ask ONE short, simple question to understand the situation better.
Do not give advice yet. Just ask. Be calm and reassuring.
Use the same language as the person — Hindi, English, or mixed.

--- PHASE 2: After 3 exchanges OR when you have enough information ---
Do BOTH at the same time:
1. Give immediate simple advice based on what you know so far
2. Ask one more follow-up question to understand better

Format for Phase 2:
Right now: [1-2 simple things they can do immediately]
Ye karo / Do this: [action]
Aur batao / Tell me more: [one follow-up question]

--- PHASE 3: Final assessment (when you have clear picture) ---
Stop asking questions. Give a clear verdict.

EMERGENCY SIGNS — if you see 2 or more confirmed:
- Not waking up and making no sounds
- Breathing stopped or very slow (confirmed, not just "collapsed")
- Face or lips blue, grey or very pale
- Chest pain with sweating or arm pain
- Unresponsive to both touch and voice
- Sudden collapse with additional symptoms confirmed
- - Elderly person (60+) suddenly confused, talking nonsense, OR one sided weakness
- Sudden severe headache with confusion or vomiting
- Face drooping or uneven smile

If EMERGENCY confirmed:
EMERGENCY - AMBULANCE BULAO ABHI / CALL AMBULANCE NOW
Jab tak ambulance aaye / While waiting:
1. [simple action]
2. [simple action]  
3. [simple action]

If NOT emergency:
DOCTOR KO DIKHAO / SEE A DOCTOR - Go within a few hours
[1-2 things to do right now while going]

OR:
MONITOR KARO / KEEP WATCHING
[1-2 things to watch for, when to escalate]

STRICT RULES:
- Never use medical words. Say "heart beating fast" not "tachycardia"
- Maximum 4 sentences per response
- Never diagnose. Never say "this is definitely X disease"
- "He collapsed" alone = ask a question, NOT emergency
- CRITICAL: Always reply in the EXACT language the person used.
- If they wrote in English → reply in English only
- If they wrote in Hindi → reply in Hindi only  
- If they wrote in mixed Hindi-English → reply in mixed Hindi-English
- Do not switch languages under any circumstance
- After 5 exchanges total, you MUST give a final assessment even if unsure"""

conversation_history = []
current_image = [None]

def get_response(user_text, image=None):
    global conversation_history
    content = []
    if image:
        content.append({"type": "image", "image": image})
        content.append({"type": "text", "text": user_text + " (photo sent)"})
    else:
        content.append({"type": "text", "text": user_text})
    conversation_history.append({"role": "user", "content": content})
    messages = [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]}] + conversation_history
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=150,
            do_sample=True, temperature=0.7, top_p=0.9
        )
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    conversation_history.append({"role": "assistant", "content": [{"type": "text", "text": response}]})
    return response

def speak(text):
    hindi_chars = sum(1 for c in text if '\u0900' <= c <= '\u097F')
    lang = "hi" if hindi_chars > 5 else "en"
    tts = gTTS(text=text, lang=lang, slow=False)
    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as f:
        tts.save(f.name)
        return f.name

# UI
display(HTML("""
<div style='background:linear-gradient(135deg,#c0392b,#e74c3c);
     padding:20px;border-radius:12px;text-align:center;margin-bottom:15px'>
    <h2 style='color:white;margin:0'>🏥 AidFirst</h2>
    <p style='color:#ffdddd;margin:5px 0 0 0;font-size:13px'>
        Medical First Response &nbsp;|&nbsp; मेडिकल फर्स्ट रिस्पांस
    </p>
</div>
"""))

chat_out = widgets.Output()
audio_out = widgets.Output()
status_out = widgets.Output()

text_input = widgets.Text(
    placeholder="Type in English or Hindi...",
    layout=widgets.Layout(width="75%", height="36px")
)
send_btn = widgets.Button(
    description="Send →", button_style="danger",
    layout=widgets.Layout(width="23%", height="36px")
)
image_upload = widgets.FileUpload(
    accept="image/*", multiple=False,
    description="📷 Photo",
    layout=widgets.Layout(width="49%", margin="5px 0")
)
audio_upload = widgets.FileUpload(
    accept="audio/*", multiple=False,
    description="🎤 Voice",
    layout=widgets.Layout(width="49%", margin="5px 0")
)
reset_btn = widgets.Button(
    description="🔄 New Assessment", button_style="warning",
    layout=widgets.Layout(width="100%", margin="5px 0")
)

def on_image_upload(change):
    if image_upload.value:
        file_info = list(image_upload.value.values())[0]
        img = Image.open(io.BytesIO(file_info["content"])).convert("RGB")
        current_image[0] = img
        with status_out:
            status_out.clear_output()
            display(HTML("<p style='color:green;margin:0'> Photo ready</p>"))

image_upload.observe(on_image_upload, names="value")

def on_send(b):
    global conversation_history
    user_text = text_input.value.strip()

    # Audio transcription
    if not user_text and audio_upload.value:
        with status_out:
            status_out.clear_output()
            display(HTML("<p style='color:gray;margin:0'> Transcribing...</p>"))
        file_info = list(audio_upload.value.values())[0]
        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as f:
            f.write(file_info["content"])
            audio_path = f.name
        result = whisper_model.transcribe(audio_path)
        user_text = result["text"]
        with chat_out:
            display(HTML(f"""
            <div style='background:#e3f2fd;padding:10px 14px;border-radius:10px;
                        margin:6px 0;text-align:right;border-left:3px solid #1976d2'>
                <span style='color:#555;font-size:12px'>You </span><br>
                <span style='font-size:14px'>{user_text}</span>
            </div>"""))

    if not user_text:
        with status_out:
            status_out.clear_output()
            display(HTML("<p style='color:red;margin:0'>Please type or upload a voice memo.</p>"))
        return

    text_input.value = ""

    with status_out:
        status_out.clear_output()
        display(HTML("<p style='color:gray;margin:0'>⏳ Thinking...</p>"))

    img = current_image[0]
    current_image[0] = None

    # Show user message (only if typed, audio already shown above)
    if not audio_upload.value:
        img_note = " 📷" if img else ""
        with chat_out:
            display(HTML(f"""
            <div style='background:#e3f2fd;padding:10px 14px;border-radius:10px;
                        margin:6px 0;text-align:right;border-left:3px solid #1976d2'>
                <span style='color:#555;font-size:12px'>You{img_note}</span><br>
                <span style='font-size:14px'>{user_text}</span>
            </div>"""))

    response = get_response(user_text, image=img)
    is_emergency = "EMERGENCY" in response
    bg = "#ffebee" if is_emergency else "#f1f8e9"
    border = "#c62828" if is_emergency else "#558b2f"
    icon = "⚠️" if is_emergency else "🏥"

    with chat_out:
        display(HTML(f"""
        <div style='background:{bg};padding:10px 14px;border-radius:10px;
                    margin:6px 0;border-left:3px solid {border}'>
            <span style='color:#555;font-size:12px'>{icon} NidarAI</span><br>
            <span style='font-size:14px'>{response.replace(chr(10), "<br>")}</span>
        </div>"""))

    audio_file = speak(response)
    with audio_out:
        audio_out.clear_output()
        display(HTML("<p style='color:gray;font-size:12px;margin:4px 0'>🔊 Audio response:</p>"))
        display(Audio(audio_file, autoplay=True))

    with status_out:
        status_out.clear_output()

def on_reset(b):
    global conversation_history
    conversation_history = []
    current_image[0] = None
    chat_out.clear_output()
    audio_out.clear_output()
    status_out.clear_output()
    with chat_out:
        display(HTML("<p style='color:#bbb;text-align:center;padding:20px'>New assessment started.</p>"))

send_btn.on_click(on_send)
reset_btn.on_click(on_reset)

display(widgets.HBox([text_input, send_btn]))
display(widgets.HBox([image_upload, audio_upload]))
display(reset_btn)
display(status_out)
display(audio_out)
display(chat_out)

with chat_out:
    display(HTML("""
    <div style='text-align:center;color:#bbb;padding:30px;font-size:13px'>
        Describe what is happening to get started.<br>
        बताएं क्या हो रहा है।
    </div>"""))

Loading Whisper...
✅ Ready. UI loading...


Button(button_style='warning', description='🔄 New Assessment', layout=Layout(margin='5px 0', width='100%'), st…

Output()

Output()

Output()